# Time series 3: Blend (α=0.91) vs Naive by one-month intervals

Evaluates the **winning blend model** (α=0.91: 0.91×naive + 0.09×seasonal) and the **naive** baseline on the same 1-step-ahead test set (last 8760 hours), but **by one-month intervals**: MAE and MASE are computed separately for each month so we can see how each model performs over time.

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

_root = Path.cwd().resolve()
if _root.name == "time_series":
    _root = _root.parent
DATA_PATH = _root / "data" / "transformed" / "transformed_data.parquet"
if not DATA_PATH.exists():
    DATA_PATH = DATA_PATH.with_suffix(".csv")
if not DATA_PATH.exists():
    raise FileNotFoundError(f"Transform data not found at {DATA_PATH}. Run task transform first.")

df = pd.read_parquet(DATA_PATH) if DATA_PATH.suffix == ".parquet" else pd.read_csv(DATA_PATH)
df = df.sort_values(["date", "hour"]).reset_index(drop=True)
df["datetime"] = pd.to_datetime(df["date"]) + pd.to_timedelta(df["hour"], unit="h")

freq_bands = sorted([c for c in df.columns if c not in ("date", "hour", "datetime")])
TEST_STEPS = 24 * 365
HOURS_PER_MONTH = TEST_STEPS // 12  # 730 hours per month

print(f"Loaded {len(df)} rows. Date range: {df['date'].min()} to {df['date'].max()}")
print(f"Frequency bands: {len(freq_bands)}")
print(f"Test window: last {TEST_STEPS} hours (~12 months, {HOURS_PER_MONTH} h/month)")

Loaded 9192 rows. Date range: 2025-01-01 to 2026-01-18
Frequency bands: 70
Test window: last 8760 hours (~12 months, 730 h/month)


## Build 1-step naive, seasonal naive, and blend (α=0.91)

Same logic as in `time_series_2.ipynb`: naive = last value, seasonal = same hour yesterday. Blend = 0.91×naive + 0.09×seasonal.

In [2]:
ALPHA = 0.91

rows_naive, rows_seasonal = [], []
for freq in freq_bands:
    sub = df[["datetime", freq]].dropna(subset=[freq]).sort_values("datetime").reset_index(drop=True)
    if len(sub) < TEST_STEPS + 24:
        continue
    vals = sub[freq].values.astype(np.float64)
    dts = sub["datetime"].values
    test_vals = vals[-TEST_STEPS:]
    test_dts = dts[-TEST_STEPS:]
    for i in range(0, TEST_STEPS - 1):
        rows_naive.append({"datetime": test_dts[i+1], "frequency_band": freq, "actual": test_vals[i+1], "predicted": test_vals[i]})
    for i in range(23, TEST_STEPS - 1):
        rows_seasonal.append({"datetime": test_dts[i+1], "frequency_band": freq, "actual": test_vals[i+1], "predicted": test_vals[i-23]})

df_naive = pd.DataFrame(rows_naive)
df_seasonal = pd.DataFrame(rows_seasonal)
df_naive = df_naive.rename(columns={"predicted": "p_naive"})[["datetime", "frequency_band", "actual", "p_naive"]]
df_seasonal = df_seasonal.rename(columns={"predicted": "p_seasonal"})[["datetime", "frequency_band", "p_seasonal"]]
merge_b = df_naive.merge(df_seasonal, on=["datetime", "frequency_band"], how="inner")
merge_b["predicted"] = ALPHA * merge_b["p_naive"] + (1 - ALPHA) * merge_b["p_seasonal"]
df_blend = merge_b[["datetime", "frequency_band", "actual", "predicted"]].copy()
df_naive = df_naive.rename(columns={"p_naive": "predicted"})

print(f"Naive: {len(df_naive)} rows; Blend (α={ALPHA}): {len(df_blend)} rows")

Naive: 613130 rows; Blend (α=0.91): 611520 rows


## Assign month index (0..11) to each test hour

Test set = last 8760 hours. Split into 12 consecutive months of 730 hours each. Each prediction row gets a `month` index (0 = first month, 11 = last month).

In [4]:
# Test datetimes in chronological order (from first band) so month index matches test window
first_freq = freq_bands[0]
sub = df[["datetime", first_freq]].dropna().sort_values("datetime").reset_index(drop=True)
if len(sub) < TEST_STEPS + 1:
    test_dates_ordered = df_naive["datetime"].drop_duplicates().sort_values().values
else:
    test_dates_ordered = sub["datetime"].values[-TEST_STEPS:]
n_test = len(test_dates_ordered)
dt_to_month = {dt: i // HOURS_PER_MONTH for i, dt in enumerate(test_dates_ordered)}

df_naive["month"] = df_naive["datetime"].map(dt_to_month)
df_blend["month"] = df_blend["datetime"].map(dt_to_month)

print(f"Test hours: {n_test}, hours per month: {HOURS_PER_MONTH}")
print("Month 0 = first 730h of test, month 11 = last 730h")

Test hours: 8760, hours per month: 730
Month 0 = first 730h of test, month 11 = last 730h


## MASE denominator

MASE = MAE / scaling. Scaling = mean absolute first difference of actuals (per band, then averaged). We compute it once over the full test set so monthly MASE is comparable.

In [5]:
diffs = df_naive.sort_values(["frequency_band", "datetime"]).groupby("frequency_band")["actual"].diff().dropna().abs()
mase_denom = float(diffs.mean()) if len(diffs) > 0 else np.nan
print(f"MASE denominator (mean |actual_t - actual_t-1|): {mase_denom:.6f}")

MASE denominator (mean |actual_t - actual_t-1|): 2.420043


## MAE and MASE per month: Naive vs Blend (α=0.91)

In [6]:
def mae_mase_for_subset(sub_df):
    if len(sub_df) == 0:
        return np.nan, np.nan
    mae = float(np.mean(np.abs(sub_df["actual"] - sub_df["predicted"])))
    mase = mae / mase_denom if mase_denom and mase_denom > 0 else np.nan
    return mae, mase

months = sorted(df_naive["month"].dropna().unique())
rows_table = []
for m in months:
    sub_naive = df_naive[df_naive["month"] == m]
    sub_blend = df_blend[df_blend["month"] == m]
    mae_n, mase_n = mae_mase_for_subset(sub_naive)
    mae_b, mase_b = mae_mase_for_subset(sub_blend)
    rows_table.append({
        "month": int(m),
        "naive_MAE": mae_n,
        "naive_MASE": mase_n,
        "blend_MAE": mae_b,
        "blend_MASE": mase_b,
        "winner": "Blend" if mae_b < mae_n else "Naive",
    })

table = pd.DataFrame(rows_table)
print("Per-month MAE and MASE (1-step-ahead):")
print(table.to_string(index=False))

mae_naive_overall = float(np.mean(np.abs(df_naive["actual"] - df_naive["predicted"])))
mae_blend_overall = float(np.mean(np.abs(df_blend["actual"] - df_blend["predicted"])))
mase_naive_overall = mae_naive_overall / mase_denom if mase_denom else np.nan
mase_blend_overall = mae_blend_overall / mase_denom if mase_denom else np.nan

print("\n--- Overall (full test set) ---")
print(f"  Naive:  MAE = {mae_naive_overall:.4f},  MASE = {mase_naive_overall:.4f}" if not np.isnan(mase_naive_overall) else f"  Naive:  MAE = {mae_naive_overall:.4f}")
print(f"  Blend (α={ALPHA}): MAE = {mae_blend_overall:.4f},  MASE = {mase_blend_overall:.4f}" if not np.isnan(mase_blend_overall) else f"  Blend (α={ALPHA}): MAE = {mae_blend_overall:.4f}")
print(f"  Blend wins {sum(1 for r in rows_table if r['winner'] == 'Blend')} / {len(rows_table)} months.")

Per-month MAE and MASE (1-step-ahead):
 month  naive_MAE  naive_MASE  blend_MAE  blend_MASE winner
     0   2.617721    1.081684   2.542425    1.050570  Blend
     1   2.298702    0.949860   2.232049    0.922318  Blend
     2   2.093139    0.864918   2.100671    0.868030  Naive
     3   1.977419    0.817101   1.971544    0.814673  Blend
     4   1.220716    0.504419   1.209339    0.499718  Blend
     5   1.309284    0.541017   1.447187    0.598001  Naive
     6   1.968165    0.813277   2.149143    0.888060  Naive
     7   0.937283    0.387300   0.934235    0.386041  Blend
     8   1.677536    0.693184   1.633952    0.675175  Blend
     9   4.192336    1.732339   3.962947    1.637552  Blend
    10   5.009162    2.069864   4.756503    1.965462  Blend
    11   3.736632    1.544035   3.744764    1.547395  Naive

--- Overall (full test set) ---
  Naive:  MAE = 2.4198,  MASE = 0.9999
  Blend (α=0.91): MAE = 2.3900,  MASE = 0.9876
  Blend wins 8 / 12 months.


## Diebold-Mariano test: Naive vs Blend (α=0.91)

Test whether the difference in MAE between naive and the winning blend is statistically significant. H0: equal predictive accuracy. We use MAE loss: *d*_t = |e_naive| − |e_blend|; DM statistic is approximately N(0,1) under H0. Two-sided *p*-value; *p* < 0.05 ⇒ reject equal accuracy. DM > 0 ⇒ naive worse (blend has lower MAE); DM < 0 ⇒ naive better.

In [7]:
from scipy import stats

def dm_test(e1, e2, loss="abs"):
    """e1, e2: arrays of forecast errors (same length). loss='abs' => MAE. Returns (dm_stat, p_value_two_sided)."""
    e1, e2 = np.asarray(e1, dtype=float), np.asarray(e2, dtype=float)
    n = len(e1)
    if n != len(e2) or n < 2:
        return np.nan, np.nan
    if loss == "abs":
        d = np.abs(e1) - np.abs(e2)
    else:
        d = (e1 ** 2) - (e2 ** 2)
    d_bar = np.mean(d)
    sigma_d = np.std(d, ddof=1)
    if sigma_d <= 0:
        return 0.0, 1.0
    dm_stat = d_bar / (sigma_d / np.sqrt(n))
    p_value = 2 * (1 - stats.norm.cdf(abs(dm_stat)))
    return float(dm_stat), float(p_value)

# Align on (datetime, frequency_band); use inner join so we have paired errors
m = df_naive.merge(
    df_blend,
    on=["datetime", "frequency_band"],
    suffixes=("_naive", "_blend"),
    how="inner",
)
e_naive = (m["actual_naive"] - m["predicted_naive"]).values
e_blend = (m["actual_blend"] - m["predicted_blend"]).values
dm_stat, p_val = dm_test(e_naive, e_blend, loss="abs")
# DM > 0 => naive has larger |errors| => blend has lower MAE
winner = "Blend" if dm_stat > 0 else "Naive"
sig = " *" if p_val < 0.05 else ""

print("Diebold-Mariano (MAE loss, two-sided). H0: equal accuracy.")
print(f"  Naive vs Blend (α={ALPHA}): n = {len(m)}, DM = {dm_stat:.4f}, p = {p_val:.4f}{sig}")
print(f"  Lower MAE: {winner}")
if p_val < 0.05:
    print("  → Reject H0: the difference in MAE is statistically significant.")
else:
    print("  → Do not reject H0: we cannot say the difference is significant at 5%.")

Diebold-Mariano (MAE loss, two-sided). H0: equal accuracy.
  Naive vs Blend (α=0.91): n = 611520, DM = 22.6987, p = 0.0000 *
  Lower MAE: Blend
  → Reject H0: the difference in MAE is statistically significant.


## Try to beat the blend: tune α on validation

Use the **last 14 days** of data before the test window as validation. Grid-search α in [0.88, 0.89, …, 0.93] and pick the α that minimizes MAE on validation. Then evaluate that α on the **test set** and compare to the fixed blend (α=0.91).

In [8]:
VAL_HOURS = 24 * 14  # 14 days validation (last 14 days before test)
alphas = np.arange(0.88, 0.935, 0.01)  # [0.88, 0.89, ..., 0.93]
best_alpha, best_mae_val = None, np.inf
val_errors_by_alpha = []

for alpha in alphas:
    mae_val = 0.0
    n_val = 0
    for freq in freq_bands:
        sub = df[["datetime", freq]].dropna(subset=[freq]).sort_values("datetime").reset_index(drop=True)
        if len(sub) < TEST_STEPS + 24 + VAL_HOURS:
            continue
        vals = sub[freq].values.astype(np.float64)
        T = len(vals) - TEST_STEPS  # train length
        # Validation: last VAL_HOURS of train; 1-step: actual at i+1, p_naive at i, p_seasonal at i-24
        for i in range(max(T - VAL_HOURS, 23), T - 1):
            actual = vals[i + 1]
            p_naive = vals[i]
            p_seasonal = vals[i - 24]
            pred = alpha * p_naive + (1 - alpha) * p_seasonal
            mae_val += np.abs(actual - pred)
            n_val += 1
    mae_val = mae_val / n_val if n_val > 0 else np.inf
    val_errors_by_alpha.append((alpha, mae_val))
    if mae_val < best_mae_val:
        best_mae_val = mae_val
        best_alpha = alpha

print(f"Validation (last {VAL_HOURS//24} days): best α = {best_alpha:.2f}, MAE_val = {best_mae_val:.4f}")
print("Grid: " + ", ".join(f"α={a:.2f}→{e:.4f}" for a, e in val_errors_by_alpha))

# Build test predictions with best_alpha (same rows as df_blend)
rows_tuned = []
for freq in freq_bands:
    sub = df[["datetime", freq]].dropna(subset=[freq]).sort_values("datetime").reset_index(drop=True)
    if len(sub) < TEST_STEPS + 24:
        continue
    vals = sub[freq].values.astype(np.float64)
    test_vals, test_dts = vals[-TEST_STEPS:], sub["datetime"].values[-TEST_STEPS:]
    for i in range(23, TEST_STEPS - 1):
        rows_tuned.append({
            "datetime": test_dts[i + 1],
            "frequency_band": freq,
            "actual": test_vals[i + 1],
            "predicted": best_alpha * test_vals[i] + (1 - best_alpha) * test_vals[i - 23],
        })
df_tuned = pd.DataFrame(rows_tuned)
mae_tuned_test = float(np.mean(np.abs(df_tuned["actual"] - df_tuned["predicted"])))
mase_tuned_test = mae_tuned_test / mase_denom if mase_denom and mase_denom > 0 else np.nan
mae_blend_test = float(np.mean(np.abs(df_blend["actual"] - df_blend["predicted"])))

print("\n--- Test set (12 months) ---")
print(f"  Blend (fixed α=0.91): MAE = {mae_blend_test:.4f}")
print(f"  Blend (tuned α={best_alpha:.2f}):  MAE = {mae_tuned_test:.4f},  MASE = {mase_tuned_test:.4f}" if not np.isnan(mase_tuned_test) else f"  Blend (tuned α={best_alpha:.2f}):  MAE = {mae_tuned_test:.4f}")
if mae_tuned_test < mae_blend_test:
    print("  → Tuned α beats fixed 0.91 on test.")
else:
    print("  → Fixed 0.91 still best (or tie).")

Validation (last 14 days): best α = 0.88, MAE_val = 2.6495
Grid: α=0.88→2.6495, α=0.89→2.6516, α=0.90→2.6541, α=0.91→2.6569, α=0.92→2.6601, α=0.93→2.6636

--- Test set (12 months) ---
  Blend (fixed α=0.91): MAE = 2.3900
  Blend (tuned α=0.88):  MAE = 2.3925,  MASE = 0.9886
  → Fixed 0.91 still best (or tie).


## Three-way blend: naive + daily + weekly seasonal

pred = **α×y(t−1) + β×y(t−24) + γ×y(t−168)** with α+β+γ=1. Weekly seasonal y(t−168) = same hour, same weekday last week. Tune (α, β, γ) on the same 14-day validation, then evaluate on test.

In [11]:
LAG168 = 168
# Grid: (α, β, γ) with α+β+γ=1; γ = weekly weight
trials = []
for alpha in [0.86, 0.88, 0.90, 0.91, 0.92]:
    for gamma in [0, 0.02, 0.04]:
        beta = 1 - alpha - gamma
        if beta < 0:
            continue
        trials.append((alpha, beta, gamma))

best_3w, best_mae_3w_val = None, np.inf
for (alpha, beta, gamma) in trials:
    mae_val = 0.0
    n_val = 0
    for freq in freq_bands:
        sub = df[["datetime", freq]].dropna(subset=[freq]).sort_values("datetime").reset_index(drop=True)
        # Need at least one validation point: T >= LAG168+2 (T = len - TEST_STEPS)
        if len(sub) < TEST_STEPS + LAG168 + 2:
            continue
        vals = sub[freq].values.astype(np.float64)
        T = len(vals) - TEST_STEPS
        # Validation: use last VAL_HOURS of train when available, else all train with 168 lags
        i_start = max(T - VAL_HOURS, LAG168)
        for i in range(i_start, T - 1):
            actual = vals[i + 1]
            pred = alpha * vals[i] + beta * vals[i - 24] + gamma * vals[i - LAG168]
            mae_val += np.abs(actual - pred)
            n_val += 1
    mae_val = mae_val / n_val if n_val > 0 else np.inf
    if mae_val < best_mae_3w_val:
        best_mae_3w_val = mae_val
        best_3w = (alpha, beta, gamma)

if best_3w is None:
    print("Three-way: no valid (α,β,γ) in grid.")
else:
    alpha_3w, beta_3w, gamma_3w = best_3w
    rows_3w = []
    for freq in freq_bands:
        sub = df[["datetime", freq]].dropna(subset=[freq]).sort_values("datetime").reset_index(drop=True)
        if len(sub) < TEST_STEPS + LAG168:
            continue
        vals = sub[freq].values.astype(np.float64)
        test_vals = vals[-TEST_STEPS:]
        test_dts = sub["datetime"].values[-TEST_STEPS:]
        for i in range(LAG168, TEST_STEPS - 1):
            rows_3w.append({
                "datetime": test_dts[i + 1],
                "frequency_band": freq,
                "actual": test_vals[i + 1],
                "predicted": alpha_3w * test_vals[i] + beta_3w * test_vals[i - 24] + gamma_3w * test_vals[i - LAG168],
            })
    df_3w = pd.DataFrame(rows_3w)
    mae_3w_test = float(np.mean(np.abs(df_3w["actual"] - df_3w["predicted"])))
    mase_3w_test = mae_3w_test / mase_denom if mase_denom and mase_denom > 0 else np.nan
    print(f"Three-way blend (validation-tuned): α={alpha_3w:.2f}, β={beta_3w:.2f}, γ={gamma_3w:.2f}")
    print(f"  Validation MAE = {best_mae_3w_val:.4f}; Test MAE = {mae_3w_test:.4f}, MASE = {mase_3w_test:.4f}" if not np.isnan(mase_3w_test) else f"  Test MAE = {mae_3w_test:.4f}")
    mae_blend_ref = float(np.mean(np.abs(df_blend["actual"] - df_blend["predicted"])))
    print(f"  Blend (α=0.91) test MAE = {mae_blend_ref:.4f}")
    if mae_3w_test < mae_blend_ref:
        print("  → Three-way blend beats two-way blend on test.")
    else:
        print("  → Two-way blend (0.91) still best or tie.")

Three-way blend (validation-tuned): α=0.86, β=0.12, γ=0.02
  Validation MAE = 2.6952; Test MAE = 2.5050, MASE = 1.0351
  Blend (α=0.91) test MAE = 2.3900
  → Two-way blend (0.91) still best or tie.
